# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

We will be using Ranking/Scoring because 

**Scoring** is the process of assigning a numerical value or probability to each data instance that indicates how likely it is to belong to a particular class, exhibit a behavior, or produce an outcome.

**Ranking** is the process of ordering data instances based on their predicted scores or relevance so that the highest-priority or most relevant items appear first.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df=pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

We will predict the priorities of webpages that needs to be refreshed first. That label come from defined rule because we dont have specific column in our datasets that tells whether the SEO expert refreshes that page or not so we bsically will use **proxy** to approximate the real outcome because we dont have the kind of dataset that can be measured or observed directly.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create boolean flags for key risk & value signals
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['is_page_one'] = (df['position_tier'] == 'page_1').astype(int)
df['is_high_value'] = (df['cpc'] > df['cpc'].median()).astype(int)
df['opportunity_score'] = ((df['is_declining'] * 35) + (df['is_page_one'] * 25) +(df['is_high_value'] * 20) +((1 - df['ctr'].fillna(0)) * 20)).round(1)
print("Proxy target constructed successfully!")
print(df['opportunity_score'].describe())

Proxy target constructed successfully!
count    30000.000000
mean        43.171333
std         70.518769
min      -1980.000000
25%         27.800000
50%         50.600000
75%         70.400000
max        100.000000
Name: opportunity_score, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric: Precision@K (Precision@15)**

Why this metric?
Standard classification accuracy is not useful here because an SEO editor cannot review thousands of pages. If an editor only has bandwidth to review 15 pages a week, Precision@15 measures what percentage of the top 15 recommended pages were actually dying, high-value pages worth their time.

What number means 'good'?
A Precision@15 score above 0.80 (80%) means at least 12 out of the top 15 pages handed to the team are urgent, high-impact refresh candidates.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_15_queue = df.sort_values(by='opportunity_score', ascending=False).head(15)
print("Top 15 Pages in the Refresh Queue:")
print(top_15_queue[['content_id', 'cpc', 'avg_position', 'trend_direction', 'opportunity_score']])


Top 15 Pages in the Refresh Queue:
                 content_id   cpc  avg_position trend_direction  \
17549  content_3474a43ad37f  0.31           4.0            down   
17611  content_370f162b9110  1.26           6.3            down   
12839  content_602f886c306c  0.19           3.5            down   
12870  content_bc94bc662bb6  0.09           8.1            down   
12948  content_f5ebddad5548  1.28           6.3            down   
20881  content_30aac2f74fd3  0.07           7.6            down   
20949  content_d7344dbc6ad1  0.21           8.8            down   
29161  content_33ddda1a8c7a  0.24           6.9            down   
20928  content_8ef8109b0801  2.37           8.0            down   
3599   content_f5b2d4a27311  0.97           8.8            down   
27579  content_ab46905e22f7  1.46           4.8            down   
17329  content_83bc621f73e2  1.61           9.7            down   
27771  content_0588cb9e1922  1.05           3.8            down   
27524  content_addafe3df89b

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: 1 row = 1 unique webpage (identified by content_id).

The dataframe below demonstrates that each row captures the metrics, engagement signals, search performance, and constructed opportunity score for a single content piece.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show unit of analysis as a clean slice
cols_to_show = ['content_id', 'main_intent','cpc','avg_position','trend_direction','opportunity_score']
print(f"Unit of Analysis DataFrame Shape: {df[cols_to_show].head().shape}")
df[cols_to_show].head()

Unit of Analysis DataFrame Shape: (5, 6)


,content_id,main_intent,cpc,avg_position,trend_direction,opportunity_score
0,content_304f48230142,transactional,2.05,10.6,down,59.8
1,content_a1fb4e703a9e,informational,0.05,20.3,down,74.0
2,content_9aa793d4d895,informational,0.00,36.5,down,53.2
3,content_331d6c4de07b,commercial,0.00,6.2,stable,35.2
4,content_d99b7a2d90ca,informational,0.00,44.0,down,52.4


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-else rule (e.g., if age > 180 and trend == 'down') fails in practice because:

It lacks nuance: A 170-day-old page losing 50,000 visits on a high-CPC keyword might be ignored by a strict 180-day rule, while a 181-day-old page losing only 2 visits gets flagged.

Multi-variable trade-offs: Ranking position, traffic decline, engagement rates, and search value interact in complex, non-linear ways. Hardcoded if-else logic quickly becomes unmaintainable, whereas ML dynamically weights competing signals simultaneously to produce an optimal review queue.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_based = df[(df['trend_direction'] == 'down') & (df['position_tier'] == 'page_1')]
print(f"Total pages flagged by basic rule: {len(rule_based)}")
print(f"Total pages in dataset: {len(df)}")
print("ML allows us to continuously score and rank all pages instead of a binary cut-off.")

Total pages flagged by basic rule: 6730
Total pages in dataset: 30000
ML allows us to continuously score and rank all pages instead of a binary cut-off.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.